# Laboratorio 9 - Deep Learning

- Juan Solís - 23720
- Victor Pérez - 23731
- Diego Flores - 23714
- Nils Muralles - 23727
- Isabella Recinos - 23003

## Task 3

### Task 3.1

Se implementa una U-Net pequeña condicionada por el instante y la clase. La clase 10 representa la condición nula.

In [1]:
import math
import torch
import torch.nn as nn

SEED = 42
torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"

class SinusoidalEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        if dim % 2 != 0:
            raise ValueError("dim debe ser par")
        frequencies = torch.exp(
            torch.arange(0, dim, 2, dtype=torch.float32)
            * (-math.log(10000.0) / dim)
        )
        self.dim = dim
        self.register_buffer("frequencies", frequencies)

    def forward(self, t):
        angles = t.float().unsqueeze(1) * self.frequencies.unsqueeze(0)
        embedding = torch.empty(t.size(0), self.dim, device=t.device)
        embedding[:, 0::2] = torch.sin(angles)
        embedding[:, 1::2] = torch.cos(angles)
        return embedding


class ConditionedBlock(nn.Module):
    def __init__(self, in_channels, out_channels, condition_dim):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.norm1 = nn.GroupNorm(8, out_channels)
        self.condition_projection = nn.Linear(condition_dim, out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.norm2 = nn.GroupNorm(8, out_channels)
        self.activation = nn.SiLU()

    def forward(self, x, condition):
        h = self.activation(self.norm1(self.conv1(x)))
        h = h + self.condition_projection(condition)[:, :, None, None]
        h = self.activation(self.norm2(self.conv2(h)))
        return h


class SmallConditionalUNet(nn.Module):
    def __init__(self, condition_dim=64):
        super().__init__()
        self.time_embedding = SinusoidalEmbedding(condition_dim)
        self.time_mlp = nn.Sequential(
            nn.Linear(condition_dim, condition_dim),
            nn.SiLU(),
            nn.Linear(condition_dim, condition_dim),
        )
        self.class_embedding = nn.Embedding(11, condition_dim)

        self.encoder_28 = ConditionedBlock(1, 32, condition_dim)
        self.downsample = nn.Conv2d(32, 64, 4, stride=2, padding=1)
        self.encoder_14 = ConditionedBlock(64, 64, condition_dim)
        self.bottleneck = ConditionedBlock(64, 64, condition_dim)

        self.upsample = nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1)
        self.decoder_28 = ConditionedBlock(64, 32, condition_dim)
        self.output = nn.Conv2d(32, 1, 1)

    def forward(self, x, t, c):
        time_condition = self.time_mlp(self.time_embedding(t))
        class_condition = self.class_embedding(c)
        condition = time_condition + class_condition

        skip = self.encoder_28(x, condition)
        h = self.downsample(skip)
        h = self.encoder_14(h, condition)
        h = self.bottleneck(h, condition)
        h = self.upsample(h)
        h = torch.cat([h, skip], dim=1)
        h = self.decoder_28(h, condition)
        return self.output(h)


model = SmallConditionalUNet().to(device)
x = torch.randn(4, 1, 28, 28, device=device)
t = torch.tensor([1, 250, 500, 1000], device=device)
c = torch.tensor([0, 3, 10, 7], device=device)

model.eval()
with torch.no_grad():
    predicted_noise = model(x, t, c)

parameter_count = sum(p.numel() for p in model.parameters())
assert predicted_noise.shape == x.shape
assert torch.isfinite(predicted_noise).all()
assert parameter_count == 272929

print("Forma de entrada:", tuple(x.shape))
print("Forma de salida:", tuple(predicted_noise.shape))
print("Parametros totales:", parameter_count)
print("Semilla:", SEED, "| device:", device)

Forma de entrada: (4, 1, 28, 28)
Forma de salida: (4, 1, 28, 28)
Parametros totales: 272929
Semilla: 42 | device: cpu
